Feature construction (engineering) creates informative inputs for models (interactions, aggregates, encodings, transforms).
Splitting separates data for honest evaluation (train / validation / test) and prevents data leakage.
Always fit preprocessing/transformers on training data only; apply to validation/test.


## Feature Construction

In [133]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [134]:
df = pd.read_csv(r"C:\Users\Acer\OneDrive\Desktop\Machine Learning\CSV\tested.csv")[['Age' , 'Pclass' , 'SibSp' , 'Parch' , 'Survived']]

In [135]:
df.dropna(inplace=True)

In [136]:
df.head()

,Age,Pclass,SibSp,Parch,Survived
0,34.5,3,0,0,0
1,47.0,3,1,0,1
2,62.0,2,0,0,0
3,27.0,3,0,0,0
4,22.0,3,1,1,1


In [137]:
x = df.drop(columns=['Survived'])
y = df['Survived']

In [138]:
x

,Age,Pclass,SibSp,Parch
0,34.5,3,0,0
1,47.0,3,1,0
2,62.0,2,0,0
3,27.0,3,0,0
4,22.0,3,1,1
...,...,...,...,...
409,3.0,3,1,1
411,37.0,1,1,0
412,28.0,3,0,0
414,39.0,1,0,0


In [139]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
np.mean(cross_val_score(LogisticRegression() , x , y , scoring='accuracy' ,cv=20))

np.float64(0.6238970588235293)

In [140]:
x['Family_Size'] = x['Parch'] + x['SibSp'] + 1

In [141]:
x.head()

,Age,Pclass,SibSp,Parch,Family_Size
0,34.5,3,0,0,1
1,47.0,3,1,0,2
2,62.0,2,0,0,1
3,27.0,3,0,0,1
4,22.0,3,1,1,3


In [142]:
def my_func(num):
    if num == 1:
        return 1
    elif num>1 and num<=4:
        return 2
    else:
        return 3
    

In [143]:
my_func(6)

3

In [144]:
x['Family_type'] = x['Family_Size'].apply(my_func)

In [145]:
x.head()

,Age,Pclass,SibSp,Parch,Family_Size,Family_type
0,34.5,3,0,0,1,1
1,47.0,3,1,0,2,2
2,62.0,2,0,0,1,1
3,27.0,3,0,0,1,1
4,22.0,3,1,1,3,2


In [146]:
x.drop(columns=['Parch',  'SibSp' , 'Family_Size'])

,Age,Pclass,Family_type
0,34.5,3,1
1,47.0,3,2
2,62.0,2,1
3,27.0,3,1
4,22.0,3,2
...,...,...,...
409,3.0,3,2
411,37.0,1,2
412,28.0,3,1
414,39.0,1,1


In [147]:
np.mean(cross_val_score(LogisticRegression() , x , y , scoring='accuracy' ,  cv=20))

np.float64(0.6305147058823529)

As we can see our accuracy is slightly increased.

## Feature Splitting

In [148]:
df = pd.read_csv(r"C:\Users\Acer\OneDrive\Desktop\Machine Learning\CSV\tested.csv")

In [149]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [150]:
df.dropna(inplace=True)

In [151]:
df['Name']

12         Snyder, Mrs. John Pillsbury (Nelle Stevenson)
14     Chaffee, Mrs. Herbert Fuller (Carrie Constance...
24       Ryerson, Mrs. Arthur Larned (Emily Maria Borie)
26                          Ostby, Miss. Helene Ragnhild
28                               Brady, Mr. John Bertram
                             ...                        
404                         Frauenthal, Mr. Isaac Gerald
405         Nourney, Mr. Alfred (Baron von Drachstedt")"
407                           Widener, Mr. George Dunton
411      Minahan, Mrs. William Edward (Lillian E Thorpe)
414                         Oliva y Ocana, Dona. Fermina
Name: Name, Length: 87, dtype: object

I want only salutation word at mr , mrs or miss

In [152]:
df['Title'] = df['Name'].str.split(',' , expand=True)[1].str.split('.' , expand=True)[0]

In [153]:
df['Title'] = df['Title'].str.strip()

In [154]:
(df.groupby('Title')['Survived'].mean()).sort_values(ascending=False)

Title
Dona      1.0
Mrs       1.0
Miss      1.0
Dr        0.0
Col       0.0
Master    0.0
Mr        0.0
Name: Survived, dtype: float64

In [155]:
def is_married(title):
    if (title == 'Mrs'):
        return 1
    else:
        return 0

In [156]:
df['Is_married'] = df['Title'].apply(is_married)

In [157]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Is_married
12,904,1,1,"Snyder, Mrs. John Pillsbury (Nelle Stevenson)",female,23.0,1,0,21228,82.2667,B45,S,Mrs,1
14,906,1,1,"Chaffee, Mrs. Herbert Fuller (Carrie Constance...",female,47.0,1,0,W.E.P. 5734,61.1750,E31,S,Mrs,1
24,916,1,1,"Ryerson, Mrs. Arthur Larned (Emily Maria Borie)",female,48.0,1,3,PC 17608,262.3750,B57 B59 B63 B66,C,Mrs,1
26,918,1,1,"Ostby, Miss. Helene Ragnhild",female,22.0,0,1,113509,61.9792,B36,C,Miss,0
28,920,0,1,"Brady, Mr. John Bertram",male,41.0,0,0,113054,30.5000,A21,S,Mr,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404,1296,0,1,"Frauenthal, Mr. Isaac Gerald",male,43.0,1,0,17765,27.7208,D40,C,Mr,0
405,1297,0,2,"Nourney, Mr. Alfred (Baron von Drachstedt"")""",male,20.0,0,0,SC/PARIS 2166,13.8625,D38,C,Mr,0
407,1299,0,1,"Widener, Mr. George Dunton",male,50.0,1,1,113503,211.5000,C80,C,Mr,0
411,1303,1,1,"Minahan, Mrs. William Edward (Lillian E Thorpe)",female,37.0,1,0,19928,90.0000,C78,Q,Mrs,1


In this way we can explore and play more with data..